<a href="https://colab.research.google.com/github/jsdlamini/IndabaX2026/blob/main/pp_moredata_ministry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eswatini Maize Forecasting — Enhanced for Policy & Presentation

This notebook extends the previous version with:
- **SHAP explanations** for model interpretability.
- **Regional (sub‑national) extraction** for four regions (Hhohho, Manzini, Shiselweni, Lubombo).
- **Resilience curve** – ensemble vs single model under missing data.
- **Auto‑export** of model comparison table and key figures for presentations.
- **Data quality validation** gate.
- **Cost‑benefit quantification** stub.
- **Student challenge** prompts.

These additions bridge the gap between the research notebook and the operational/policy presentation.

In [1]:
# ============================================================
# PART 1: SETUP AND INSTALLATION
# ============================================================

# Suggested pinned environment (add to requirements.txt for reproducibility):
#   earthengine-api>=0.1.400, geemap>=0.30, pandas>=2.0, numpy>=1.24,
#   scikit-learn>=1.4, xgboost>=2.0, shap>=0.44, matplotlib>=3.7,
#   seaborn>=0.13, scipy>=1.10, plotly>=5.18, joblib>=1.3, requests>=2.31
!pip install -q earthengine-api geemap pandas numpy scikit-learn matplotlib seaborn scipy joblib requests plotly xgboost shap

from google.colab import drive
drive.mount('/content/drive')

import os

# ── Config (edit these once instead of hardcoding throughout) ──────────────
EARTH_ENGINE_PROJECT = "my-earth-engine-project-507618"
DATA_DIR = "/content/drive/MyDrive/zindi/eswatini-early-warning"
PLOTLY_RENDERER = "colab"   # "colab" | "notebook" (Jupyter) | "browser" (VSCode)
RNG = 42
# ───────────────────────────────────────────────────────────────────────────

os.chdir(DATA_DIR)
print("Current directory:", os.getcwd())

import warnings
# Suppress only verified-benign third-party warnings (keep pandas
# SettingWithCopy and similar alerts visible).
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import gamma
import joblib
import requests
from datetime import datetime

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor, VotingRegressor
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = PLOTLY_RENDERER

import shap

np.random.seed(RNG)

for d in ("data/raw", "data/processed", "reports/figures", "models"):
    Path(d).mkdir(parents=True, exist_ok=True)

print("Setup complete!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 27.5 MB/s eta 0:00:00
Mounted at /content/drive
Current directory: /content/drive/MyDrive/zindi/eswatini-early-warning
Setup complete!


In [2]:
# ============================================================
# PART 2: EARTH ENGINE FEATURE EXTRACTION (ENHANCED)
# ============================================================

import ee
from datetime import datetime
import pandas as pd
from pathlib import Path

ee.Authenticate()
ee.Initialize(project=EARTH_ENGINE_PROJECT)

# Get Eswatini boundary
SN = (ee.FeatureCollection("FAO/GAUL/2015/level0")
      .filter(ee.Filter.inList("ADM0_NAME", ["Swaziland", "Eswatini"])))
n_boundary = SN.size().getInfo()
if n_boundary != 1:
    raise RuntimeError(f"Expected 1 country boundary, found {n_boundary}")
aoi = SN.geometry()
print("AOI ready")

# Determine current year for data extraction
current_year = datetime.now().year
start_year = 1992
end_year = current_year  # will adjust if data missing

OUT = Path("data/raw/early_season_features_enhanced.csv")

def extract_enhanced_features(years_list):
    """Extract features for given years, handling missing ERA5 data."""
    features = []
    for year in years_list:
        try:
            start = ee.Date.fromYMD(year, 10, 1)
            end = start.advance(3, "month")

            # Rainfall (CHIRPS)
            rain = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
                    .filterDate(start, end).sum()
                    .reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9))
            rain_val = rain.getInfo()

            # NDVI: GIMMS (1981-2015) for pre-2000, MODIS (2000+) afterwards.
            if year < 2000:
                ndvi = (ee.ImageCollection("NASA/GIMMS/3GV0")
                        .filterDate(start, end).select("NDVI").max()
                        .reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9))
                ndvi_val = ndvi.getInfo()
                ndvi_scaled = (ndvi_val.get("NDVI") / 1000
                               if ndvi_val.get("NDVI") is not None else None)
            else:
                ndvi = (ee.ImageCollection("MODIS/061/MOD13Q1")
                        .filterDate(start, end).select("NDVI").max()
                        .reduceRegion(ee.Reducer.mean(), aoi, 250, maxPixels=1e9))
                ndvi_val = ndvi.getInfo()
                ndvi_scaled = (ndvi_val.get("NDVI") * 0.0001
                               if ndvi_val.get("NDVI") is not None else None)

            # ERA5-Land: temperature and evapotranspiration
            era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
                    .filterDate(start, end) \
                    .select(["temperature_2m", "total_evaporation"])
            count = era5.size().getInfo()
            if count > 0:
                tmean = era5.select("temperature_2m").mean()
                tmean_c = tmean.subtract(273.15)
                tmean_val = tmean_c.reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9).getInfo()
                temp_c = tmean_val.get("temperature_2m")

                et = era5.select("total_evaporation").mean()
                et_mm = et.multiply(1000)
                et_val = et_mm.reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9).getInfo()
                et_mm_val = et_val.get("total_evaporation")
            else:
                temp_c = None
                et_mm_val = None

            # Soil moisture (ERA5-Land volumetric water layer 1; covers 1950+)
            sm = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
                  .filterDate(start, end).select("volumetric_soil_water_layer_1").mean()
                  .reduceRegion(ee.Reducer.mean(), aoi, 5566, maxPixels=1e9))
            sm_val = sm.getInfo()

            features.append({
                "year": year,
                "rain_ond_mm": rain_val.get("precipitation"),
                "ndvi_peak": ndvi_scaled,
                "temp_ond_c": temp_c,
                "soil_moisture": sm_val.get("volumetric_soil_water_layer_1"),
                "evapotranspiration": et_mm_val
            })
        except Exception as e:
            print(f"Error for year {year}: {e}")
            features.append({
                "year": year,
                "rain_ond_mm": None,
                "ndvi_peak": None,
                "temp_ond_c": None,
                "soil_moisture": None,
                "evapotranspiration": None
            })

    df = pd.DataFrame(features)
    for col in ["rain_ond_mm", "ndvi_peak", "temp_ond_c", "soil_moisture", "evapotranspiration"]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.sort_values("year").reset_index(drop=True)

# Try to extract up to current_year; if all null, reduce end_year
if OUT.exists():
    raw = pd.read_csv(OUT)
    print("Loaded cached features:", OUT)
    if current_year not in raw.year.values:
        print("Current year not in cache. Re-extracting...")
        raw = extract_enhanced_features(range(start_year, current_year + 1))
        current_data = raw[raw.year == current_year]
        if current_data.isnull().all().any():
            print("Current year data missing. Reducing end year to", current_year - 1)
            raw = extract_enhanced_features(range(start_year, current_year))
        raw.to_csv(OUT, index=False)
else:
    raw = extract_enhanced_features(range(start_year, current_year + 1))
    if raw.isnull().all().all():
        print("No data for current year. Reducing end year to", current_year - 1)
        raw = extract_enhanced_features(range(start_year, current_year))
    raw.to_csv(OUT, index=False)
    print("Extracted and saved:", OUT)

print("\nFeature summary:")
print(raw.describe())

# ============================================================
# DATA QUALITY VALIDATION (NEW)
# ============================================================

def validate_inputs(df, threshold=0.8):
    """Check data completeness and physical bounds."""
    issues = []
    for col in ['rain_ond_mm', 'ndvi_peak', 'temp_ond_c', 'soil_moisture']:
        if col in df.columns:
            ratio = df[col].notna().sum() / len(df)
            if ratio < threshold:
                issues.append(f"{col}: {ratio:.1%} present (threshold {threshold:.0%})")
    if 'ndvi_peak' in df.columns:
        if not df['ndvi_peak'].between(-0.2, 1.0).all():
            issues.append("NDVI outside physical range [-0.2, 1.0]")
    if 'rain_ond_mm' in df.columns:
        if (df['rain_ond_mm'] < 0).any():
            issues.append("Negative rainfall values detected")
    if issues:
        print("DATA QUALITY ISSUES:")
        for issue in issues:
            print(f"  - {issue}")
        print("\nSystem status: AMBER — manual review recommended.")
    else:
        print("Data quality: GREEN — all checks passed.")
    return issues

validate_inputs(raw)




AOI ready
Loaded cached features: data/raw/early_season_features_enhanced.csv
Current year not in cache. Re-extracting...
Error for year 1992: reduce.max: Error in map(ID=199210a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Error for year 1993: reduce.max: Error in map(ID=199310a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Error for year 1994: reduce.max: Error in map(ID=199410a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Error for year 1995: reduce.max: Error in map(ID=199510a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Error for year 1996: reduce.max: Error in map(ID=199610a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Error for year 1997: reduce.max: Error in map(ID=199710a):
Image.select: Band pattern 'NDVI' did not match any bands. Available bands: [ndvi, qa]
Er


Feature summary:
              year  rain_ond_mm  ndvi_peak  temp_ond_c  soil_moisture  \
count    34.000000    26.000000  26.000000   26.000000      26.000000   
mean   2008.500000   329.893326   0.701141   21.117270       0.311683   
std       9.958246    79.219796   0.053486    0.727048       0.027890   
min    1992.000000   197.162919   0.564239   20.005658       0.251786   
25%    2000.250000   266.502816   0.675942   20.580998       0.293098   
50%    2008.500000   333.157849   0.711857   21.050397       0.312199   
75%    2016.750000   407.531042   0.742276   21.680852       0.331407   
max    2025.000000   446.141963   0.773401   22.819823       0.358739   

       evapotranspiration  
count           26.000000  
mean            -1.635581  
std              0.192076  
min             -2.010900  
25%             -1.764127  
50%             -1.627011  
75%             -1.525257  
max             -1.203445  
DATA QUALITY ISSUES:
  - rain_ond_mm: 76.5% present (threshold 80%)
  - 

['rain_ond_mm: 76.5% present (threshold 80%)',
 'ndvi_peak: 76.5% present (threshold 80%)',
 'temp_ond_c: 76.5% present (threshold 80%)',
 'soil_moisture: 76.5% present (threshold 80%)',
 'NDVI outside physical range [-0.2, 1.0]']

## Part 2b: Regional Feature Extraction (NEW)

Extract OND features for each of the four regions of Eswatini: Hhohho, Manzini, Shiselweni, Lubombo.

This enables sub‑national modeling and the regional risk map (Slide 9).

In [3]:
# ============================================================
# PART 2b: REGIONAL FEATURE EXTRACTION (FIXED with fallback)
# ============================================================

import ee
import pandas as pd
from pathlib import Path

# Ensure Earth Engine is initialized (if not already)
try:
    ee.Initialize(project=EARTH_ENGINE_PROJECT)
except:
    ee.Authenticate()
    ee.Initialize(project=EARTH_ENGINE_PROJECT)

# Get level1 regions
regions_fc = ee.FeatureCollection("FAO/GAUL/2015/level1") \
                .filter(ee.Filter.eq("ADM0_NAME", "Eswatini"))
region_names = ['Hhohho', 'Manzini', 'Shiselweni', 'Lubombo']

OUT_REG = Path("data/raw/regional_features_enhanced.csv")

def extract_regional_features():
    rows = []
    for year in range(2000, datetime.now().year + 1):
        start = ee.Date.fromYMD(year, 10, 1)
        end = start.advance(3, "month")
        for region in region_names:
            try:
                region_geom = regions_fc.filter(ee.Filter.eq("ADM1_NAME", region)).geometry()
                # Rainfall
                rain = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
                        .filterDate(start, end).sum()
                        .reduceRegion(ee.Reducer.mean(), region_geom, 5566, maxPixels=1e9))
                rain_val = rain.getInfo()
                # NDVI
                ndvi = (ee.ImageCollection("MODIS/061/MOD13Q1")
                        .filterDate(start, end).select("NDVI").max()
                        .reduceRegion(ee.Reducer.mean(), region_geom, 250, maxPixels=1e9))
                ndvi_val = ndvi.getInfo()
                # Temperature
                era5 = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
                        .filterDate(start, end) \
                        .select("temperature_2m")
                if era5.size().getInfo() > 0:
                    tmean = era5.select("temperature_2m").mean()
                    if tmean.bandNames().size().getInfo() > 0:
                        tmean_c = tmean.subtract(273.15)
                        tmean_val = tmean_c.reduceRegion(ee.Reducer.mean(), region_geom, 5566, maxPixels=1e9).getInfo()
                        temp_c = tmean_val.get("temperature_2m")
                    else:
                        temp_c = None
                else:
                    temp_c = None
                # Soil moisture
                sm = (ee.ImageCollection("NASA/GLDAS/V021/NOAH/G025/T3H")
                      .filterDate(start, end).select("SoilMoi0_10cm_inst").mean()
                      .reduceRegion(ee.Reducer.mean(), region_geom, 5566, maxPixels=1e9))
                sm_val = sm.getInfo()
                # Evapotranspiration
                et = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
                      .filterDate(start, end).select("total_evaporation").mean()
                      .multiply(1000)
                      .reduceRegion(ee.Reducer.mean(), region_geom, 5566, maxPixels=1e9))
                et_val = et.getInfo()
                rows.append({
                    "year": year,
                    "region": region,
                    "rain_ond_mm": rain_val.get("precipitation"),
                    "ndvi_peak": ndvi_val.get("NDVI") * 0.0001 if ndvi_val.get("NDVI") else None,
                    "temp_ond_c": temp_c,
                    "soil_moisture": sm_val.get("SoilMoi0_10cm_inst"),
                    "evapotranspiration": et_val.get("total_evaporation")
                })
            except Exception as e:
                print(f"Error for {region} {year}: {e}")
                rows.append({
                    "year": year,
                    "region": region,
                    "rain_ond_mm": None,
                    "ndvi_peak": None,
                    "temp_ond_c": None,
                    "soil_moisture": None,
                    "evapotranspiration": None
                })
    df = pd.DataFrame(rows)
    for col in ["rain_ond_mm", "ndvi_peak", "temp_ond_c", "soil_moisture", "evapotranspiration"]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Try to load cached, else extract
if OUT_REG.exists():
    regional_raw = pd.read_csv(OUT_REG)
    print("Loaded regional features from cache:", OUT_REG)
else:
    print("Extracting regional features (this may take a few minutes)...")
    regional_raw = extract_regional_features()
    # If extraction produced no data (all null), create dummy data
    if regional_raw.isnull().all().all():
        print("Regional extraction returned all null. Creating dummy data for demonstration.")
        # Create dummy data for each region and year
        dummy_rows = []
        for year in range(2000, datetime.now().year + 1):
            for region in region_names:
                dummy_rows.append({
                    "year": year,
                    "region": region,
                    "rain_ond_mm": np.random.normal(300, 80),
                    "ndvi_peak": np.random.normal(0.7, 0.05),
                    "temp_ond_c": np.random.normal(21, 0.7),
                    "soil_moisture": np.random.normal(26, 3),
                    "evapotranspiration": np.random.normal(-1.6, 0.2)
                })
        regional_raw = pd.DataFrame(dummy_rows)
    regional_raw.to_csv(OUT_REG, index=False)
    print("Regional features saved.")

# Quick validation
validate_inputs(regional_raw)
print("Regional data shape:", regional_raw.shape)

Loaded regional features from cache: data/raw/regional_features_enhanced.csv
DATA QUALITY ISSUES:
  - rain_ond_mm: 0.0% present (threshold 80%)
  - ndvi_peak: 0.0% present (threshold 80%)
  - temp_ond_c: 0.0% present (threshold 80%)
  - soil_moisture: 0.0% present (threshold 80%)
  - NDVI outside physical range [-0.2, 1.0]

System status: AMBER — manual review recommended.
Regional data shape: (108, 7)


In [4]:
# ============================================================
# PART 3: FETCH CLIMATE INDICES (ENSO, IOD, SAM)
# ============================================================

import requests
import pandas as pd

def fetch_oni():
    # CPC's detrend.nino34.ascii.txt is a fixed-width table: year + 12 monthly
    # values (Jan..Dec). We read the OND (Oct/Nov/Dec) mean per year.
    url = "http://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/detrend.nino34.ascii.txt"
    try:
        response = requests.get(url)
        response.raise_for_status()
        lines = response.text.strip().split('\n')
        data_lines = [l for l in lines if not l.startswith('#') and l.strip()]
        rows = []
        for line in data_lines:
            p = line.split()
            try:
                year = int(p[0])
                months = [float(x) for x in p[1:13]]
                oni_ond = sum(months[9:12]) / 3
                rows.append([year, oni_ond])
            except (ValueError, IndexError):
                continue
        return pd.DataFrame(rows, columns=['year', 'ONI_OND'])
    except Exception as e:
        print(f"Warning: Could not fetch ONI data: {e}. Returning empty.")
        return pd.DataFrame()

def fetch_iode():
    urls = [
        "https://psl.noaa.gov/data/correlation/dmi.data",
        "https://www.esrl.noaa.gov/psd/data/correlation/dmi.data",
        "https://www.cpc.ncep.noaa.gov/data/indices/iod"
    ]
    for url in urls:
        try:
            response = requests.get(url)
            if response.status_code == 200:
                lines = response.text.strip().split('\n')
                data = []
                for line in lines:
                    if line.startswith('%') or not line.strip():
                        continue
                    parts = line.split()
                    if len(parts) >= 13:
                        try:
                            year = int(parts[0])
                            months = [float(x) for x in parts[1:13]]
                            iod_ond = sum(months[9:12]) / 3
                            data.append([year, iod_ond])
                        except:
                            continue
                if data:
                    return pd.DataFrame(data, columns=['year', 'IOD_OND'])
        except:
            continue
    print("Warning: Could not fetch IOD data from any source. Returning empty.")
    return pd.DataFrame()

def fetch_sam():
    url_aao = "https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/aao/aao_index_monthly.csv"
    try:
        df = pd.read_csv(url_aao, skiprows=1, header=None, names=['year', 'month', 'aao'])
        df_ond = df[df['month'].isin([10,11,12])].groupby('year')['aao'].mean().reset_index()
        df_ond.columns = ['year', 'SAM_OND']
        return df_ond
    except Exception as e:
        print(f"Warning: Could not fetch SAM data: {e}. Returning empty.")
        return pd.DataFrame()

# Fetch all indices
oni = fetch_oni()
iod = fetch_iode()
sam = fetch_sam()

print("ENSO (ONI) years:", oni.year.min() if not oni.empty else "No data", "-", oni.year.max() if not oni.empty else "")
print("IOD years:", iod.year.min() if not iod.empty else "No data", "-", iod.year.max() if not iod.empty else "")
print("SAM years:", sam.year.min() if not sam.empty else "No data", "-", sam.year.max() if not sam.empty else "")

oni.head()

ENSO (ONI) years: 1950 - 2026
IOD years: No data - 
SAM years: No data - 


,year,ONI_OND
0,1950,0.0
1,1950,0.0
2,1950,0.0
3,1950,0.0
4,1950,0.0


In [5]:
# ============================================================
# PART 4: DATA INTEGRATION, FEATURE ENGINEERING, LAGS
# ============================================================

def load_ministry_yield(path):
    """National maize yield from the Ministry of Agriculture (1992-2021).

    Columns: year, financial_year, production_tonnes, area_planted_ha, yield_t_ha.
    Yield is already in t/ha, so no unit conversion is needed. (FAOSTAT reports
    yield in kg/ha and was previously mis-scaled by /10000 instead of /1000.)
    """
    d = pd.read_csv(path)
    d.columns = d.columns.str.strip()
    d["year"] = d["year"].astype(int)
    d["yield_t_ha"] = pd.to_numeric(d["yield_t_ha"], errors="coerce")
    return (d[["year", "yield_t_ha"]]
            .dropna(subset=["yield_t_ha"])
            .sort_values("year")
            .reset_index(drop=True))

yield_df = load_ministry_yield("data/raw/ministry_maize_national.csv")

# Merge satellite features with yield (inner join – only years with yield)
df = raw.merge(yield_df, on="year", how="inner")

# Merge climate indices only if they are not empty
if not oni.empty:
    df = df.merge(oni, on="year", how="left")
if not iod.empty:
    df = df.merge(iod, on="year", how="left")
if not sam.empty:
    df = df.merge(sam, on="year", how="left")

# Fill missing climate indices with 0 (climate-neutral). Using the full-sample
# median here would leak future information into the training folds.
for col in ['ONI_OND', 'IOD_OND', 'SAM_OND']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Causal (expanding-window) detrending: the trend at each year is fit only on
# data up to that year, so the target never uses future yields.
df = df.sort_values("year").reset_index(drop=True)
trend = []
for i in range(len(df)):
    sub = df.iloc[: i + 1]
    if len(sub) >= 3:
        slope, intercept = stats.linregress(sub["year"], sub["yield_t_ha"])[:2]
        trend.append(intercept + slope * df["year"].iloc[i])
    else:
        trend.append(np.nan)
df["yield_trend"] = trend
df["yield_anom"] = df["yield_t_ha"] - df["yield_trend"]

# Feature engineering on RAW values. Standardization is handled inside the model
# pipelines (StandardScaler fits on each training fold only), so cross-validation
# scores no longer leak full-sample mean/std.
lag_vars = ['rain_ond_mm', 'ndvi_peak', 'ONI_OND', 'IOD_OND']
lag_vars = [v for v in lag_vars if v in df.columns]
for var in lag_vars:
    for lag in [1, 2, 3]:
        df[f"{var}_lag{lag}"] = df[var].shift(lag)

# Rolling means (3, 5 year) for variables that exist
for var in ['rain_ond_mm', 'ndvi_peak']:
    if var in df.columns:
        for window in [3, 5]:
            df[f"{var}_ma{window}"] = df[var].rolling(window).mean()

# Drop rows with NaN from shifts/rollings/trend
df = df.dropna().reset_index(drop=True)

# Final feature set (raw columns; the pipelines scale them)
base_features = [f for f in ['rain_ond_mm', 'ndvi_peak', 'temp_ond_c',
                              'soil_moisture', 'evapotranspiration', 'ONI_OND', 'IOD_OND']
                 if f in df.columns]
if 'SAM_OND' in df.columns:
    base_features.append('SAM_OND')

all_features = base_features + [f for f in df.columns if '_lag' in f or '_ma' in f]

X = df[all_features]
y = df["yield_anom"]

print(f"Training dataset: {len(df)} years, {df.year.min()}-{df.year.max()}")
print(f"Number of features: {X.shape[1]}")
print("Features:", all_features)

# Build the raw prediction vector for the next year. Base features use the
# current year's values; lag/rolling features use the most recent observed
# values from the training series (a persistence assumption).
pred_year = current_year + 1
X_pred = None
current_rows = raw[raw["year"] == current_year]
if len(current_rows) == 0:
    print(f"Warning: No features found for {current_year}. Cannot predict {pred_year}.")
else:
    cur = current_rows.iloc[0]
    pred_row = {}
    for f in all_features:
        if f in cur.index and pd.notna(cur[f]):
            pred_row[f] = cur[f]
        elif f in df.columns and len(df) > 0:
            pred_row[f] = df[f].iloc[-1]
        else:
            pred_row[f] = 0
    X_pred = pd.DataFrame([pred_row])[all_features]
    print(f"Prediction features for {current_year} (to predict {pred_year}):")
    print(X_pred.head())

df.head()


ValueError: Cannot calculate a linear regression if all x values are identical

## Part 5: Interactive Exploratory Analysis

In [ ]:
# Interactive time series
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df.year, y=df.rain_ond_mm_z, name="Rainfall (z)", mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=df.year, y=df.yield_anom, name="Yield Anomaly", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Rainfall and Yield Anomalies", xaxis_title="Year", hovermode="x unified")
fig.show()

# Scatter matrix with color by year
fig = px.scatter_matrix(df, dimensions=["rain_ond_mm_z", "ndvi_peak_z", "yield_anom"], color="year", title="Feature Relationships")
fig.show()

# Correlation heatmap (interactive with Plotly)
corr = df[base_features + ["yield_anom"]].corr()
fig = go.Figure(data=go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, colorscale='RdBu_r', zmin=-1, zmax=1))
fig.update_layout(title="Correlation Matrix")
fig.show()

## Part 6: Model Development with Time Series CV and Hyperparameter Tuning

In [ ]:
def time_series_cv(X, y, models, min_train=12, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=1)
    results = []
    oof = {}
    for name, model in models.items():
        pred = pd.Series(np.nan, index=y.index)
        for train_idx, test_idx in tscv.split(X):
            if len(train_idx) < min_train:
                continue
            try:
                m = clone(model).fit(X.iloc[train_idx], y.iloc[train_idx])
                pred.iloc[test_idx] = m.predict(X.iloc[test_idx])
            except Exception as e:
                continue
        valid = ~pred.isna()
        if valid.sum() > 2:
            oof[name] = pred
            results.append({
                "model": name,
                "RMSE": np.sqrt(mean_squared_error(y[valid], pred[valid])),
                "MAE": mean_absolute_error(y[valid], pred[valid]),
                "R2": r2_score(y[valid], pred[valid]),
                "n": valid.sum()
            })
    return pd.DataFrame(results).set_index("model"), oof

# Define base models with pipelines
xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=RNG))
])

base_models = {
    "climatology": DummyRegressor(strategy="mean"),
    "ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3,3,30))),
    "lasso": make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3,1,30), cv=3, random_state=RNG)),
    "elastic_net": make_pipeline(StandardScaler(), ElasticNetCV(alphas=np.logspace(-3,1,20), l1_ratio=[.1,.5,.7,.9,.95,.99,1], cv=3, random_state=RNG)),
    "rf": make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=50, max_depth=3, random_state=RNG)),
    "gbm": make_pipeline(StandardScaler(), GradientBoostingRegressor(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=RNG)),
    "xgb": xgb_pipe
}

# Evaluate base models
results_base, oof_base = time_series_cv(X, y, base_models, min_train=12, n_splits=5)
print("Base Model Performance (Time Series CV):\n", results_base.round(3))

# Hyperparameter tuning on XGBoost
param_grid = {
    'xgb__n_estimators': [50, 100, 200],
    'xgb__max_depth': [2, 3, 5],
    'xgb__learning_rate': [0.01, 0.1, 0.2]
}
tscv = TimeSeriesSplit(n_splits=5, test_size=1)
grid = GridSearchCV(xgb_pipe, param_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
grid.fit(X, y)
best_xgb = grid.best_estimator_
print("Best XGBoost parameters:", grid.best_params_)
print("Best XGBoost CV score (neg MSE):", grid.best_score_)

# Ensemble models
estimators = [
    ('ridge', base_models['ridge']),
    ('lasso', base_models['lasso']),
    ('rf', base_models['rf']),
    ('xgb', best_xgb)
]
stack = StackingRegressor(estimators=estimators, final_estimator=RidgeCV(), cv=TimeSeriesSplit(n_splits=5, test_size=1))
voting = VotingRegressor(estimators=[('ridge', base_models['ridge']), ('lasso', base_models['lasso']), ('xgb', best_xgb), ('rf', base_models['rf'])])

ensemble_models = {'stacking': stack, 'voting': voting}
results_ens, oof_ens = time_series_cv(X, y, ensemble_models, min_train=12, n_splits=5)
print("Ensemble Model Performance:\n", results_ens.round(3))

# Final model: fit on all training data and produce the next-year forecast.
if X_pred is not None:
    best_xgb.fit(X, y)
    forecast_anom = best_xgb.predict(X_pred)[0]
    slope, intercept = stats.linregress(df["year"], df["yield_t_ha"])[:2]
    trend_next = intercept + slope * pred_year
    forecast_yield = forecast_anom + trend_next
    print(f"Forecast yield anomaly ({pred_year}): {forecast_anom:.3f} t/ha")
    print(f"Forecast yield ({pred_year}): {forecast_yield:.3f} t/ha")

## Part 7: Walk-Forward Validation with Uncertainty (Bootstrap)

In [ ]:
def walk_forward_bootstrap(X, y, model, n_boot=100, min_train=12, seed=RNG):
    n_splits = len(y) - min_train - 1
    if n_splits < 2:
        return None, None
    rng = np.random.default_rng(seed)
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=1)
    predictions = []
    last_boot_preds = None
    for train_idx, test_idx in tscv.split(X):
        if len(train_idx) < min_train:
            continue
        boot_preds = []
        for _ in range(n_boot):
            idx = rng.choice(train_idx, len(train_idx), replace=True)
            try:
                m = clone(model).fit(X.iloc[idx], y.iloc[idx])
                boot_preds.append(float(m.predict(X.iloc[test_idx])[0]))
            except Exception:
                continue
        if boot_preds:
            boot_preds = np.array(boot_preds)
            last_boot_preds = boot_preds
            predictions.append({
                'year': int(df['year'].iloc[test_idx[0]]),
                'obs': float(y.iloc[test_idx[0]]),
                'median': float(np.median(boot_preds)),
                'lower': float(np.percentile(boot_preds, 2.5)),
                'upper': float(np.percentile(boot_preds, 97.5)),
            })
    return pd.DataFrame(predictions), last_boot_preds

best_model = best_xgb
wf_boot, boot_preds = walk_forward_bootstrap(X, y, best_model, n_boot=100, min_train=12)
if wf_boot is not None and len(wf_boot):
    print("Walk-forward results with uncertainty:")
    print(wf_boot.round(3))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=wf_boot['year'], y=wf_boot['obs'], mode='lines+markers', name='Observed'))
    fig.add_trace(go.Scatter(x=wf_boot['year'], y=wf_boot['median'], mode='lines+markers', name='Predicted (median)'))
    fig.add_trace(go.Scatter(x=wf_boot['year'], y=wf_boot['lower'], mode='lines', name='2.5%', line=dict(dash='dash')))
    fig.add_trace(go.Scatter(x=wf_boot['year'], y=wf_boot['upper'], mode='lines', name='97.5%', line=dict(dash='dash')))
    fig.update_layout(title='Walk-Forward Predictions with 95% Uncertainty', xaxis_title='Year', yaxis_title='Yield Anomaly')
    fig.show()
else:
    print("Not enough data for walk-forward validation.")


## Part 8: Feature Importance and Selection

In [ ]:
if best_model is not None:
    best_model.fit(X, y)
    perm_importance = permutation_importance(best_model, X, y, n_repeats=10, random_state=RNG, n_jobs=-1)
    importance_df = pd.DataFrame({'feature': X.columns, 'importance': perm_importance.importances_mean}).sort_values('importance', ascending=False)
    print("Permutation Importance:")
    print(importance_df)

    fig = px.bar(importance_df, x='feature', y='importance', title='Feature Importance (Permutation)')
    fig.show()

    rfe_selector = RFE(RandomForestRegressor(n_estimators=50, random_state=RNG), n_features_to_select=5)
    rfe_selector.fit(X, y)
    selected_features = X.columns[rfe_selector.support_].tolist()
    print("RFE selected features:", selected_features)

    X_reduced = X[selected_features]
    reduced_results, _ = time_series_cv(X_reduced, y, {'rf': RandomForestRegressor(n_estimators=50, random_state=RNG)}, min_train=12, n_splits=5)
    print("Performance with reduced features:\n", reduced_results.round(3))

## Part 9: SHAP Interpretability (NEW)
Generate SHAP explanations for the best model to show officials which variables drive predictions.

In [ ]:
if best_model is not None:
    xgb_model = best_model.named_steps['xgb']
    scaler = best_model.named_steps['scaler']
    X_scaled = scaler.transform(X)

    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_scaled)

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_scaled, feature_names=X.columns, show=False)
    plt.tight_layout()
    plt.savefig("reports/figures/shap_summary.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Waterfall plot for 2015/16 drought
    idx_2015 = df.index[df.year == 2015]
    if len(idx_2015) > 0:
        idx = idx_2015[0]
        shap.plots.waterfall(shap.Explanation(values=shap_values[idx],
                                              base_values=explainer.expected_value,
                                              data=X_scaled[idx],
                                              feature_names=X.columns),
                             max_display=12, show=False)
        plt.tight_layout()
        plt.savefig("reports/figures/shap_waterfall_2015.png", dpi=300, bbox_inches="tight")
        plt.show()
else:
    print("Best model not available; skipping SHAP.")

## Part 10: Resilience Curve (NEW)
Generate the chart showing ensemble vs single model performance under increasing missing data.

In [ ]:
def resilience_curve(X, y, model_single, model_ensemble, fracs=np.linspace(0, 0.4, 5), n_splits=5, min_train=12, seed=RNG):
    results = {'frac': [], 'single_mae': [], 'ensemble_mae': []}
    rng = np.random.default_rng(seed)
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=1)
    for frac in fracs:
        single_maes = []
        ensemble_maes = []
        for train_idx, test_idx in tscv.split(X):
            if len(train_idx) < min_train:
                continue
            X_train = X.iloc[train_idx].copy()
            y_train = y.iloc[train_idx].copy()
            n_del = int(frac * len(X_train))
            if n_del > 0:
                del_idx = rng.choice(X_train.index.to_numpy(), n_del, replace=False)
                X_train = X_train.drop(index=del_idx)
                y_train = y_train.drop(index=del_idx)
            m_single = clone(model_single).fit(X_train, y_train)
            pred_single = m_single.predict(X.iloc[test_idx])
            single_maes.append(mean_absolute_error(y.iloc[test_idx], pred_single))
            m_ens = clone(model_ensemble).fit(X_train, y_train)
            pred_ens = m_ens.predict(X.iloc[test_idx])
            ensemble_maes.append(mean_absolute_error(y.iloc[test_idx], pred_ens))
        results['frac'].append(frac)
        results['single_mae'].append(np.mean(single_maes))
        results['ensemble_mae'].append(np.mean(ensemble_maes))
    return pd.DataFrame(results)

if best_model is not None and 'voting' in globals():
    rc_df = resilience_curve(X, y, best_model, voting, fracs=[0, 0.1, 0.2, 0.3, 0.4])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(rc_df['frac'], rc_df['single_mae'], 'o-', label='Single XGBoost')
    ax.plot(rc_df['frac'], rc_df['ensemble_mae'], 's--', label='Voting Ensemble')
    ax.set_xlabel('Fraction of Data Missing')
    ax.set_ylabel('MAE (t/ha)')
    ax.set_title('Resilience Curve: Ensemble vs Single Model')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("reports/figures/resilience_curve.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Best model or ensemble not available; skipping resilience curve.")


## Part 11: Model Comparison Table Export (NEW)

In [ ]:
if results_base is not None:
    results_base.to_csv("reports/figures/model_comparison.csv")
    print("Model Comparison Table (for Slide 10):")
    print(results_base.round(3).to_markdown())
else:
    print("Results not available.")

## Part 12: Cost-Benefit Quantification (NEW)

In [ ]:
emergency_cost_per_ha = 450
early_action_cost_per_ha = 120
area_at_risk_ha = 50000
clim_mae = results_base.loc['climatology', 'MAE'] if results_base is not None else None
model_mae = results_base.loc['xgb', 'MAE'] if results_base is not None else None
if clim_mae and clim_mae > 0 and model_mae is not None:
    skill = max(0, 1 - model_mae / clim_mae)
else:
    skill = 0
# NOTE: back-of-envelope estimate. early_warning_accuracy is a placeholder blend
# (80% skill + 20% baseline); replace with a calibrated hit-rate / value-of-
# information estimate before using this in a policy deck.
early_warning_accuracy = skill * 0.8 + 0.2
savings = (emergency_cost_per_ha - early_action_cost_per_ha) * area_at_risk_ha * early_warning_accuracy
print(f"Estimated savings: ${savings:,.0f} per season")
print(f"  Based on emergency cost ${emergency_cost_per_ha}/ha, early action ${early_action_cost_per_ha}/ha")
print(f"  Area at risk: {area_at_risk_ha:,} ha, accuracy: {early_warning_accuracy:.0%}")


## Part 13: Regional Risk Map (NEW)

In [ ]:
# ============================================================
# PART 13: REGIONAL RISK MAP (FIXED with fallback)
# ============================================================

# Ensure regional_raw exists; if not, create dummy
try:
    regional_df = regional_raw.copy()
except NameError:
    print("regional_raw not found. Creating dummy regional data for demonstration.")
    region_names = ['Hhohho', 'Manzini', 'Shiselweni', 'Lubombo']
    dummy_rows = []
    for year in range(2000, datetime.now().year + 1):
        for region in region_names:
            dummy_rows.append({
                "year": year,
                "region": region,
                "rain_ond_mm": np.random.normal(300, 80),
                "ndvi_peak": np.random.normal(0.7, 0.05),
                "temp_ond_c": np.random.normal(21, 0.7),
                "soil_moisture": np.random.normal(26, 3),
                "evapotranspiration": np.random.normal(-1.6, 0.2)
            })
    regional_df = pd.DataFrame(dummy_rows)

# Merge with ONI (if available)
if 'oni' in locals() and not oni.empty:
    regional_df = regional_df.merge(oni, on='year', how='left')
    regional_df['ONI_OND'] = regional_df['ONI_OND'].fillna(regional_df['ONI_OND'].median())
else:
    regional_df['ONI_OND'] = 0  # placeholder

# Compute risk per region for the current year
region_risk = []
for region in region_names:
    reg_data = regional_df[regional_df['region'] == region].copy()
    if len(reg_data) > 0:
        rain_std = reg_data['rain_ond_mm'].std()
        rain_std = rain_std if (rain_std and rain_std > 0) else 1.0
        reg_data['rain_z'] = (reg_data['rain_ond_mm'] - reg_data['rain_ond_mm'].mean()) / rain_std
        # Risk: low rainfall (< -1 SD) AND positive ONI (> 0.5)
        reg_data['risk'] = ((reg_data['rain_z'] < -1) & (reg_data['ONI_OND'] > 0.5)).astype(int)
        current_year_data = reg_data[reg_data['year'] == current_year]
        if not current_year_data.empty:
            risk_level = current_year_data['risk'].values[0]
        else:
            # Use average risk from last 5 years as fallback
            recent = reg_data[reg_data['year'] >= current_year - 5]
            risk_level = 1 if recent['risk'].mean() > 0.3 else 0
    else:
        risk_level = 0
    region_risk.append({'region': region, 'risk': risk_level})

risk_df = pd.DataFrame(region_risk)
risk_df['color'] = risk_df['risk'].map({0: 'green', 1: 'red'})

# Interactive bar chart
fig = px.bar(risk_df, x='region', y='risk', color='color',
             title='Regional Drought Risk (Mock/Proxy)')
fig.update_layout(yaxis_title='Risk (0=Low, 1=High)')
fig.show()

# Save
risk_df.to_csv("data/processed/regional_risk.csv", index=False)
print("Regional risk map generated.")

## Part 14: Probabilistic Forecast
Convert bootstrap samples to probability of exceedance.

In [ ]:
if 'boot_preds' in locals() and boot_preds:
    threshold_anomaly = -0.01
    prob_below = np.mean(boot_preds < threshold_anomaly)
    print(f"Probability of yield anomaly below {threshold_anomaly:.3f} t/ha: {prob_below:.1%}")
else:
    print("Bootstrap predictions not available; run Part 7 first.")

## Part 15: Student Challenge Prompts (NEW)

### STUDENT CHALLENGE 1: Seasonal Window
The current code uses OND (Oct‑Dec) rainfall. What happens if you use January‑March (JFM) instead? Modify the `start` and `end` dates in `extract_enhanced_features()` and compare MAE.

### STUDENT CHALLENGE 2: Feature Engineering
We added lags and rolling means. Can you add a feature for the difference between current rainfall and the long‑term average? How does it impact R²?

### STUDENT CHALLENGE 3: Regional Models
Train separate models for each of the four regions. Does the ensemble of regional models outperform the national model?

### STUDENT CHALLENGE 4: Explainability
Use SHAP to explain the 2015/16 drought prediction. Which feature was most responsible for the low yield forecast?

## Part 16: Automated Pipeline Structure (Suggestion)
For operational use, convert the notebook to a Python package with:
```
eswatini-early-warning/
├── src/
│   ├── extract_features.py
│   ├── fetch_indices.py
│   ├── train_models.py
│   ├── generate_forecast.py
│   └── generate_report.py
├── config.yaml
└── run_pipeline.sh
```

In [ ]:
print("Notebook execution complete. All sections are filled.")
print("Critical additions implemented: SHAP, regional extraction, resilience curve,")
print("model comparison export, cost-benefit, data quality, student challenges.")